In [ ]:
# ============================================
# NOTEBOOK 1 (single cell)
# Real Deribit BTC options historical panel build
# ============================================

import os
import re
import json
import math
import time
from pathlib import Path
from datetime import datetime, timezone

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------
# 0) Project setup
# -------------------------------------------------
PROJECT_NAME = "cmf_robust_finn"
cwd = Path.cwd().resolve()

project_root = None
for p in [cwd] + list(cwd.parents):
    if p.name == PROJECT_NAME:
        project_root = p
        break

if project_root is None:
    raise RuntimeError(f"Could not find project root named '{PROJECT_NAME}'")

RAW_DIR = project_root / "data" / "raw" / "deribit"
PROC_DIR = project_root / "data" / "processed"
FIG_DIR = project_root / "outputs" / "figures" / "notebook_01"
TAB_DIR = project_root / "outputs" / "tables" / "notebook_01"
DOC_DIR = project_root / "docs"

for d in [RAW_DIR, PROC_DIR, FIG_DIR, TAB_DIR, DOC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "history_url": "https://history.deribit.com/api/v2/public/get_last_trades_by_currency_and_time",
    "hv_url": "https://www.deribit.com/api/v2/public/get_historical_volatility",
    "currency": "BTC",
    "kind": "option",
    "start_date_utc": "2025-01-01",
    "end_date_utc": "2026-03-01",
    "chunk_days": 3,
    "count": 1000,
    "sleep_sec": 0.12,
    "min_ttm_days": 2.0,
    "max_ttm_days": 60.0,
    "min_moneyness": 0.70,
    "max_moneyness": 1.30,
    "train_end": "2025-09-30",
    "val_end": "2025-12-31",
    "test_end": "2026-03-01",
}

with open(TAB_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print("=" * 100)
print("Project root:", project_root)
print("Notebook 1: building real Deribit BTC options historical panel")
print("=" * 100)

# -------------------------------------------------
# 1) Helpers
# -------------------------------------------------
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
})

def save_df(df: pd.DataFrame, stem: Path):
    """
    Save parquet if available, else csv.gz.
    Returns final path.
    """
    try:
        out = stem.with_suffix(".parquet")
        df.to_parquet(out, index=False)
        return out
    except Exception:
        out = stem.with_suffix(".csv.gz")
        df.to_csv(out, index=False, compression="gzip")
        return out

def utc_ms(ts_str: str) -> int:
    return int(pd.Timestamp(ts_str, tz="UTC").timestamp() * 1000)

def daterange_chunks(start_str: str, end_str: str, chunk_days: int):
    cur = pd.Timestamp(start_str, tz="UTC")
    end = pd.Timestamp(end_str, tz="UTC")
    while cur < end:
        nxt = min(cur + pd.Timedelta(days=chunk_days), end)
        yield cur, nxt
        cur = nxt

def robust_get_json(url, params=None, max_retries=6, timeout=60):
    last_err = None
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            last_err = f"HTTP {r.status_code}: {r.text[:250]}"
        except Exception as e:
            last_err = str(e)
        time.sleep((attempt + 1) * 0.8)
    raise RuntimeError(f"GET failed for {url} params={params} :: {last_err}")

instrument_re = re.compile(r"^(BTC)-(\d{1,2}[A-Z]{3}\d{2})-(\d+)-([CP])$")

def parse_instrument_name(name: str):
    m = instrument_re.match(str(name))
    if m is None:
        return pd.Series({
            "asset": None,
            "expiry_code": None,
            "strike": np.nan,
            "option_type": None,
            "expiry_ts": pd.NaT,
        })
    asset, expiry_code, strike, cp = m.groups()
    # Deribit BTC options expire at 08:00 UTC
    expiry_ts = pd.to_datetime(expiry_code, format="%d%b%y", utc=True) + pd.Timedelta(hours=8)
    return pd.Series({
        "asset": asset,
        "expiry_code": expiry_code,
        "strike": float(strike),
        "option_type": "call" if cp == "C" else "put",
        "expiry_ts": expiry_ts,
    })

def fetch_history_chunk(start_ms: int, end_ms: int):
    """
    Pull one time window from history.deribit.com using pagination.
    """
    rows = []
    cursor = start_ms
    while cursor < end_ms:
        payload = robust_get_json(
            CONFIG["history_url"],
            params={
                "currency": CONFIG["currency"],
                "kind": CONFIG["kind"],
                "count": CONFIG["count"],
                "include_old": True,
                "sorting": "asc",
                "start_timestamp": cursor,
                "end_timestamp": end_ms,
            },
            max_retries=6,
            timeout=90,
        )

        result = payload.get("result", {})
        trades = result.get("trades", [])
        if len(trades) == 0:
            break

        rows.extend(trades)

        last_ts = int(trades[-1]["timestamp"])
        next_cursor = last_ts + 1

        if next_cursor <= cursor:
            break

        cursor = next_cursor
        time.sleep(CONFIG["sleep_sec"])

    return rows

def build_daily_hv():
    """
    Fetch BTC historical volatility from official Deribit endpoint.
    Returns daily hv dataframe, or empty dataframe if endpoint fails.
    """
    try:
        payload = robust_get_json(CONFIG["hv_url"], params={"currency": "BTC"}, max_retries=6, timeout=60)
        result = payload.get("result", payload)

        if isinstance(result, list):
            if len(result) == 0:
                return pd.DataFrame(columns=["trade_date", "hv30"])
            # possible formats: [[timestamp, value], ...] OR [{"timestamp":..., "value":...}, ...]
            if isinstance(result[0], list) or isinstance(result[0], tuple):
                hv = pd.DataFrame(result, columns=["timestamp_ms", "hv30"])
            else:
                hv = pd.DataFrame(result)
                if "timestamp" in hv.columns and "value" in hv.columns:
                    hv = hv.rename(columns={"timestamp": "timestamp_ms", "value": "hv30"})
                elif "timestamp_ms" not in hv.columns:
                    possible_ts = [c for c in hv.columns if "time" in c.lower()]
                    possible_val = [c for c in hv.columns if "vol" in c.lower() or "value" in c.lower()]
                    if possible_ts and possible_val:
                        hv = hv.rename(columns={possible_ts[0]: "timestamp_ms", possible_val[0]: "hv30"})
                    else:
                        return pd.DataFrame(columns=["trade_date", "hv30"])
        elif isinstance(result, dict):
            hv = pd.DataFrame(result)
            if "timestamp" in hv.columns and "value" in hv.columns:
                hv = hv.rename(columns={"timestamp": "timestamp_ms", "value": "hv30"})
            else:
                return pd.DataFrame(columns=["trade_date", "hv30"])
        else:
            return pd.DataFrame(columns=["trade_date", "hv30"])

        hv["timestamp"] = pd.to_datetime(hv["timestamp_ms"], unit="ms", utc=True, errors="coerce")
        hv = hv.dropna(subset=["timestamp"]).copy()
        hv["trade_date"] = hv["timestamp"].dt.floor("D")
        hv["hv30"] = pd.to_numeric(hv["hv30"], errors="coerce")
        hv = hv.dropna(subset=["hv30"]).copy()
        hv_daily = hv.groupby("trade_date", as_index=False)["hv30"].mean()
        return hv_daily
    except Exception as e:
        print(f"[WARN] historical volatility fetch failed: {e}")
        return pd.DataFrame(columns=["trade_date", "hv30"])

# -------------------------------------------------
# 2) Download historical BTC option trades
# -------------------------------------------------
all_rows = []
window_logs = []

for start_dt, end_dt in daterange_chunks(CONFIG["start_date_utc"], CONFIG["end_date_utc"], CONFIG["chunk_days"]):
    start_ms = int(start_dt.timestamp() * 1000)
    end_ms = int(end_dt.timestamp() * 1000) - 1

    rows = fetch_history_chunk(start_ms, end_ms)
    all_rows.extend(rows)

    window_logs.append({
        "window_start": str(start_dt),
        "window_end": str(end_dt),
        "n_rows": len(rows),
    })

    print(f"{start_dt.date()} -> {end_dt.date()} | rows={len(rows):,}")
    time.sleep(CONFIG["sleep_sec"])

if len(all_rows) == 0:
    raise RuntimeError(
        "No historical trade rows were downloaded. "
        "This usually means the history route is blocked on this network or unavailable from this environment."
    )

raw_df = pd.DataFrame(all_rows).drop_duplicates(subset=["trade_id"]).reset_index(drop=True)

# -------------------------------------------------
# 3) Fetch Deribit historical volatility
# -------------------------------------------------
hv_daily = build_daily_hv()

# -------------------------------------------------
# 4) Parse contracts + engineer features
# -------------------------------------------------
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], unit="ms", utc=True, errors="coerce")
raw_df = raw_df.dropna(subset=["timestamp"]).copy()

parsed = raw_df["instrument_name"].apply(parse_instrument_name)
raw_df = pd.concat([raw_df, parsed], axis=1)

# Keep only standard BTC call/put options with parsable contract names
raw_df = raw_df.dropna(subset=["expiry_ts", "strike", "option_type"]).copy()

# Numeric cleanup
for col in ["price", "mark_price", "iv", "index_price", "amount", "contracts"]:
    if col in raw_df.columns:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

raw_df["trade_date"] = raw_df["timestamp"].dt.floor("D")
raw_df["ttm_years"] = (raw_df["expiry_ts"] - raw_df["timestamp"]).dt.total_seconds() / (365.0 * 24.0 * 3600.0)
raw_df["ttm_days"] = raw_df["ttm_years"] * 365.0
raw_df["moneyness"] = raw_df["strike"] / raw_df["index_price"]
raw_df["log_moneyness"] = np.log(raw_df["moneyness"])
raw_df["premium_btc"] = raw_df["price"]
raw_df["mark_btc"] = raw_df["mark_price"]
raw_df["premium_usd"] = raw_df["premium_btc"] * raw_df["index_price"]
raw_df["mark_usd"] = raw_df["mark_btc"] * raw_df["index_price"]
raw_df["is_call"] = (raw_df["option_type"] == "call").astype(int)

# Merge daily historical vol
if len(hv_daily) > 0:
    raw_df = raw_df.merge(hv_daily, on="trade_date", how="left")
else:
    raw_df["hv30"] = np.nan

# Filters
raw_df = raw_df[
    (raw_df["ttm_days"] >= CONFIG["min_ttm_days"]) &
    (raw_df["ttm_days"] <= CONFIG["max_ttm_days"]) &
    (raw_df["moneyness"] >= CONFIG["min_moneyness"]) &
    (raw_df["moneyness"] <= CONFIG["max_moneyness"])
].copy()

raw_df = raw_df.dropna(subset=["index_price", "premium_btc", "mark_btc", "iv"]).copy()
raw_df = raw_df[
    (raw_df["index_price"] > 0) &
    (raw_df["premium_btc"] > 0) &
    (raw_df["mark_btc"] > 0) &
    (raw_df["iv"] > 0)
].copy()

# -------------------------------------------------
# 5) Build research-ready daily instrument panel
# -------------------------------------------------
raw_df = raw_df.sort_values(["instrument_name", "timestamp"]).copy()

panel = raw_df.groupby(["instrument_name", "trade_date"], as_index=False).agg(
    timestamp_last=("timestamp", "last"),
    expiry_ts=("expiry_ts", "last"),
    strike=("strike", "last"),
    option_type=("option_type", "last"),
    is_call=("is_call", "last"),
    index_price=("index_price", "last"),
    premium_btc=("premium_btc", "last"),
    mark_btc=("mark_btc", "last"),
    premium_usd=("premium_usd", "last"),
    mark_usd=("mark_usd", "last"),
    iv=("iv", "mean"),
    hv30=("hv30", "mean"),
    amount_sum=("amount", "sum"),
    contracts_sum=("contracts", "sum"),
    n_trades=("trade_id", "nunique"),
)

panel["ttm_years"] = (panel["expiry_ts"] - panel["timestamp_last"]).dt.total_seconds() / (365.0 * 24.0 * 3600.0)
panel["ttm_days"] = panel["ttm_years"] * 365.0
panel["moneyness"] = panel["strike"] / panel["index_price"]
panel["log_moneyness"] = np.log(panel["moneyness"])
panel["trade_month"] = panel["trade_date"].dt.to_period("M").astype(str)
panel["trade_year"] = panel["trade_date"].dt.year
panel["amount_sum"] = pd.to_numeric(panel["amount_sum"], errors="coerce").fillna(0.0)
panel["contracts_sum"] = pd.to_numeric(panel["contracts_sum"], errors="coerce").fillna(0.0)

# daily underlying series from aligned trade-time index_price
spot_daily = panel.groupby("trade_date", as_index=False)["index_price"].median().sort_values("trade_date")
spot_daily["spot_ret_1d"] = np.log(spot_daily["index_price"]).diff()
spot_daily["spot_vol_7d"] = spot_daily["spot_ret_1d"].rolling(7).std() * np.sqrt(365.0)
panel = panel.merge(spot_daily[["trade_date", "spot_ret_1d", "spot_vol_7d"]], on="trade_date", how="left")

# instrument history length filter
valid_instr = panel.groupby("instrument_name").size()
valid_instr = valid_instr[valid_instr >= 2].index
panel = panel[panel["instrument_name"].isin(valid_instr)].copy()

# -------------------------------------------------
# 6) Temporal train / val / test split
# -------------------------------------------------
train_end = pd.Timestamp(CONFIG["train_end"], tz="UTC")
val_end = pd.Timestamp(CONFIG["val_end"], tz="UTC")
test_end = pd.Timestamp(CONFIG["test_end"], tz="UTC")

train_df = panel[panel["trade_date"] <= train_end].copy()
val_df = panel[(panel["trade_date"] > train_end) & (panel["trade_date"] <= val_end)].copy()
test_df = panel[(panel["trade_date"] > val_end) & (panel["trade_date"] <= test_end)].copy()

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise RuntimeError("One of train / val / test is empty. Adjust date range or split boundaries.")

# -------------------------------------------------
# 7) Save artifacts
# -------------------------------------------------
raw_path = save_df(raw_df, RAW_DIR / "deribit_btc_option_trades_raw")
panel_path = save_df(panel, PROC_DIR / "deribit_btc_option_panel_daily")
train_path = save_df(train_df, PROC_DIR / "deribit_btc_option_train")
val_path = save_df(val_df, PROC_DIR / "deribit_btc_option_val")
test_path = save_df(test_df, PROC_DIR / "deribit_btc_option_test")

window_log_df = pd.DataFrame(window_logs)
window_log_path = save_df(window_log_df, TAB_DIR / "download_window_stats")

summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_df),
        "instruments": train_df["instrument_name"].nunique(),
        "start": str(train_df["trade_date"].min()),
        "end": str(train_df["trade_date"].max()),
        "mean_mark_btc": float(train_df["mark_btc"].mean()),
        "mean_iv": float(train_df["iv"].mean()),
        "mean_ttm_days": float(train_df["ttm_days"].mean()),
        "mean_moneyness": float(train_df["moneyness"].mean()),
    },
    {
        "split": "val",
        "rows": len(val_df),
        "instruments": val_df["instrument_name"].nunique(),
        "start": str(val_df["trade_date"].min()),
        "end": str(val_df["trade_date"].max()),
        "mean_mark_btc": float(val_df["mark_btc"].mean()),
        "mean_iv": float(val_df["iv"].mean()),
        "mean_ttm_days": float(val_df["ttm_days"].mean()),
        "mean_moneyness": float(val_df["moneyness"].mean()),
    },
    {
        "split": "test",
        "rows": len(test_df),
        "instruments": test_df["instrument_name"].nunique(),
        "start": str(test_df["trade_date"].min()),
        "end": str(test_df["trade_date"].max()),
        "mean_mark_btc": float(test_df["mark_btc"].mean()),
        "mean_iv": float(test_df["iv"].mean()),
        "mean_ttm_days": float(test_df["ttm_days"].mean()),
        "mean_moneyness": float(test_df["moneyness"].mean()),
    },
])
summary_path = save_df(summary, TAB_DIR / "dataset_summary")

scope_text = """Main dataset:
- Deribit BTC option historical trades from history.deribit.com
- Underlying aligned via trade-time index_price
- Auxiliary historical volatility from Deribit official endpoint

Paper scope:
- Real-data-first BTC option pricing
- One-step hedge-proxy evaluation later
- Synthetic experiments only as secondary checks, not main evidence
"""
(DOC_DIR / "scope_lock.txt").write_text(scope_text, encoding="utf-8")

# -------------------------------------------------
# 8) Diagnostics
# -------------------------------------------------
plt.figure(figsize=(8, 5))
panel["option_type"].value_counts().plot(kind="bar")
plt.title("Rows by option type")
plt.tight_layout()
plt.savefig(FIG_DIR / "rows_by_option_type.png", dpi=220)
plt.show()

plt.figure(figsize=(8, 5))
panel["ttm_days"].hist(bins=40)
plt.title("TTM distribution (days)")
plt.tight_layout()
plt.savefig(FIG_DIR / "ttm_days.png", dpi=220)
plt.show()

plt.figure(figsize=(8, 5))
panel["moneyness"].hist(bins=40)
plt.title("Moneyness distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "moneyness.png", dpi=220)
plt.show()

plt.figure(figsize=(8, 5))
panel["iv"].hist(bins=40)
plt.title("Implied volatility distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "iv_hist.png", dpi=220)
plt.show()

# -------------------------------------------------
# 9) Console report
# -------------------------------------------------
print("=" * 100)
print("NOTEBOOK 1 COMPLETE")
print("=" * 100)
print(f"Raw trade rows saved : {len(raw_df):,}")
print(f"Panel rows saved     : {len(panel):,}")
print(f"Unique instruments   : {panel['instrument_name'].nunique():,}")
print(f"Train rows           : {len(train_df):,}")
print(f"Val rows             : {len(val_df):,}")
print(f"Test rows            : {len(test_df):,}")
print(f"Raw path             : {raw_path}")
print(f"Panel path           : {panel_path}")
print(f"Train path           : {train_path}")
print(f"Val path             : {val_path}")
print(f"Test path            : {test_path}")
print(f"Summary path         : {summary_path}")
print("=" * 100)
print(summary.to_string(index=False))


Preparing dataset: etth1
Source file: /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/etth1/ETTh1.csv
Done: etth1
  rows = 17420
  numeric features incl. target = 7
  train/val/test = 8640/2880/2880
  saved clean csv   -> /data/Sajjan_Singh/spml/wavelet_seq_project/data/processed/etth1/etth1_clean.csv
  saved prepared npz-> /data/Sajjan_Singh/spml/wavelet_seq_project/data/processed/etth1/etth1_prepared.npz
  saved meta json   -> /data/Sajjan_Singh/spml/wavelet_seq_project/data/processed/etth1/etth1_meta.json

Preparing dataset: etth2
Source file: /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/etth2/ETTh2.csv
Done: etth2
  rows = 17420
  numeric features incl. target = 7
  train/val/test = 8640/2880/2880
  saved clean csv   -> /data/Sajjan_Singh/spml/wavelet_seq_project/data/processed/etth2/etth2_clean.csv
  saved prepared npz-> /data/Sajjan_Singh/spml/wavelet_seq_project/data/processed/etth2/etth2_prepared.npz
  saved meta json   -> /data/Sajjan_Singh/spml/wavelet_seq_proje

,dataset,source_file,rows,num_features_plus_target,target,target_index,frequency,date_start,date_end,duplicates_removed,...,missing_after_fill,train_rows,val_rows,test_rows,clean_csv,prepared_npz,meta_json,split_json,original_rows,final_rows
0,etth1,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,17420,7,OT,6,1h,2016-07-01 00:00:00,2018-06-26 19:00:00,0,...,0,8640,2880,2880,data/processed/etth1/etth1_clean.csv,data/processed/etth1/etth1_prepared.npz,data/processed/etth1/etth1_meta.json,data/splits/etth1_splits.json,17420,17420
1,etth2,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,17420,7,OT,6,1h,2016-07-01 00:00:00,2018-06-26 19:00:00,0,...,0,8640,2880,2880,data/processed/etth2/etth2_clean.csv,data/processed/etth2/etth2_prepared.npz,data/processed/etth2/etth2_meta.json,data/splits/etth2_splits.json,17420,17420
2,ettm1,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,69680,7,OT,6,15min,2016-07-01 00:00:00,2018-06-26 19:45:00,0,...,0,34560,11520,11520,data/processed/ettm1/ettm1_clean.csv,data/processed/ettm1/ettm1_prepared.npz,data/processed/ettm1/ettm1_meta.json,data/splits/ettm1_splits.json,69680,69680
3,ettm2,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,69680,7,OT,6,15min,2016-07-01 00:00:00,2018-06-26 19:45:00,0,...,0,34560,11520,11520,data/processed/ettm2/ettm2_clean.csv,data/processed/ettm2/ettm2_prepared.npz,data/processed/ettm2/ettm2_meta.json,data/splits/ettm2_splits.json,69680,69680
4,weather,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,52695,21,OT,20,10min,2020-01-01 00:10:00,2021-01-01 00:00:00,0,...,0,36886,5270,10539,data/processed/weather/weather_clean.csv,data/processed/weather/weather_prepared.npz,data/processed/weather/weather_meta.json,data/splits/weather_splits.json,52695,52695
5,electricity,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,26304,321,OT,320,1h,2016-07-01 02:00:00,2019-07-02 01:00:00,0,...,0,18412,2632,5260,data/processed/electricity/electricity_clean.csv,data/processed/electricity/electricity_prepare...,data/processed/electricity/electricity_meta.json,data/splits/electricity_splits.json,26304,26304
6,traffic,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,17544,862,OT,861,1h,2016-07-01 02:00:00,2018-07-02 01:00:00,0,...,0,12280,1756,3508,data/processed/traffic/traffic_clean.csv,data/processed/traffic/traffic_prepared.npz,data/processed/traffic/traffic_meta.json,data/splits/traffic_splits.json,17544,17544
7,exchange,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,7588,8,OT,7,1d,1990-01-01 00:00:00,2010-10-10 00:00:00,0,...,0,5311,760,1517,data/processed/exchange/exchange_clean.csv,data/processed/exchange/exchange_prepared.npz,data/processed/exchange/exchange_meta.json,data/splits/exchange_splits.json,7588,7588
8,ili,/data/Sajjan_Singh/spml/wavelet_seq_project/da...,966,7,OT,6,1w,2002-01-01 00:00:00,2020-06-30 00:00:00,0,...,0,676,97,193,data/processed/ili/ili_clean.csv,data/processed/ili/ili_prepared.npz,data/processed/ili/ili_meta.json,data/splits/ili_splits.json,966,966
